In [ ]:
import pandas as pd
import ast

In [ ]:
movies_df=pd.read_csv("../DATA/RAW/tmdb_5000_movies.csv")
credits_df=pd.read_csv("../DATA/RAW/tmdb_5000_credits.csv")

In [ ]:
movies_df.columns

In [ ]:
credits_df.columns

In [ ]:
movies_df=movies_df.rename(columns={"id":"movie_id"})

In [ ]:
movie_df=pd.merge(movies_df,credits_df,on="movie_id",how="outer")

In [ ]:
movie_df.columns

In [ ]:
movie_df.drop(columns=["title_x"],inplace=True)
movie_df.drop(columns=["title_y"],inplace=True)

In [ ]:
import pdfplumber
import pandas as pd
import re

pdf_path = "../DATA/RAW/Bollywood Movie Details.pdf"

text_data = []
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            lines = text.split("\n")
            text_data.extend(lines)

movies = []
current_movie = {}
collecting_overview = False

for line in text_data:
    line = line.strip()

    # Detect Movie Title
    if line and not line.startswith("•"):
        if current_movie:
            movies.append(current_movie)
            current_movie = {}
        current_movie["Title"] = line
        collecting_overview = False

    # Detect key-value lines
    elif line.startswith("•") and ":" in line:
        line = line.replace("•", "").strip()
        key, value = line.split(":", 1)
        key = key.strip()
        value = value.strip()

        # Normalize key names
        key = re.sub(r"\s*\(.*?\)", "", key).strip()

        if key == "Overview":
            current_movie[key] = value
            collecting_overview = True
        else:
            current_movie[key] = value
            collecting_overview = False

    # Handle multi-line Overview
    elif collecting_overview and line:
        current_movie["Overview"] += " " + line

if current_movie:
    movies.append(current_movie)

df_movies = pd.DataFrame(movies)

# Rename columns
rename_map = {
    "Original Language": "original_language",
    "Original Title": "original_title",
    "Production Companies": "production_companies",
    "Production Countries": "production_countries",
    "Release Date": "release_date",
    "Revenue": "revenue",
    "Runtime": "runtime",
    "Spoken Languages": "spoken_languages",
    "Crew": "crew",
    "Cast":"cast",
    "Tags":"tags",
    "Genres":"genres",
    "Budget":"budget",
    "Status":"status",
    "Overview":"overview"
}
df_movies.rename(columns=rename_map, inplace=True)

# Export to Excel
df_movies.to_excel("../DATA/RAW/Bollywood_Movie_Details.xlsx", index=False)

# Display the DataFrame
pd.set_option("display.max_colwidth", None)
display(df_movies)


In [ ]:
# Group by 'Title' and merge rows by taking first non-null value from each column
import numpy as np
df_merged = (
    df_movies
    .groupby("Title", dropna=False, sort=False)
    .agg(lambda x: x.dropna().iloc[0] if x.notna().any() else np.nan)
    .reset_index()
)

display(df_merged)

In [ ]:
df_merged = df_merged.rename(columns={'Title': 'original_title'})
df_merged = df_merged.loc[:, ~df_merged.columns.duplicated()]
union_df = pd.concat([movie_df, df_merged], ignore_index=True)
movie_df=union_df

In [ ]:
movie_df["runtime"].mode()[0]
movie_df["runtime"]=movie_df["runtime"].fillna(movie_df["runtime"].mode()[0])

In [ ]:
movie_df["tagline"].fillna("unknown",inplace=True)

In [ ]:
# movie_df.fillna("Not Available",inplace=True)
text_cols = movie_df.select_dtypes(include="object").columns

movie_df[text_cols] = movie_df[text_cols].fillna("Not Available")

num_cols = movie_df.select_dtypes(include=["int64", "float64"]).columns

movie_df[num_cols] = movie_df[num_cols].fillna(0)

In [ ]:
# movie_df["genres"]=movie_df["genres"].apply(lambda x: ",".join([d["name"] for d in ast.literal_eval(x)]))
import ast

def clean_genres(x):
    # Case 1: Proper JSON-like string
    try:
        data = ast.literal_eval(x)
        if isinstance(data, list):
            return ",".join([d["name"] for d in data if isinstance(d, dict) and "name" in d])
    except:
        pass

    # Case 2: Already comma-separated string
    if isinstance(x, str) and "," in x:
        return ",".join([i.strip() for i in x.split(",")])

    # Case 3: Single word (e.g., "Comedy")
    if isinstance(x, str):
        return x.strip()

    return ""
movie_df["genres"] = movie_df["genres"].apply(clean_genres)
movie_df["genres"]

In [ ]:
# movie_df['cast'] = movie_df['cast'].apply(lambda x: ast.literal_eval(x))
# movie_df['cast'] = movie_df['cast'].apply(
#     lambda cast_list: ", ".join([c['name'] for c in cast_list])
# )
import ast

def clean_cast(x):
    # Case 1: JSON-like string
    try:
        data = ast.literal_eval(x)
        if isinstance(data, list):
            return ", ".join([
                d["name"] for d in data 
                if isinstance(d, dict) and "name" in d
            ])
    except:
        pass

    # Case 2: Already a comma-separated string
    if isinstance(x, str) and "," in x:
        return ", ".join([i.strip() for i in x.split(",")])

    # Case 3: Single name or unknown
    if isinstance(x, str):
        return x.strip()

    return ""
movie_df["cast"] = movie_df["cast"].apply(clean_cast)
movie_df["cast"]

In [ ]:
# movie_df['crew'] = movie_df['crew'].apply(lambda x: ast.literal_eval(x))
# # Now convert the list of dicts to comma-separated names
# movie_df['crew'] = movie_df['crew'].apply(lambda crew_list: ", ".join([c['name'] for c in crew_list]))
import ast

def clean_crew(x):
    # Case 1: JSON-like string
    try:
        data = ast.literal_eval(x)
        if isinstance(data, list):
            return ", ".join([
                d["name"] for d in data 
                if isinstance(d, dict) and "name" in d
            ])
    except:
        pass

    # Case 2: Already comma-separated string
    if isinstance(x, str) and "," in x:
        return ", ".join([i.strip() for i in x.split(",")])

    # Case 3: Single name
    if isinstance(x, str):
        return x.strip()

    return ""
movie_df["crew"] = movie_df["crew"].apply(clean_crew)
movie_df["crew"]


In [ ]:
# movie_df["keywords"] = movie_df["keywords"].apply(
#     lambda x: ",".join(d["name"] for d in ast.literal_eval(x))
# )
import ast

def clean_keywords(x):
    # Case 1: JSON-like string
    try:
        data = ast.literal_eval(x)
        if isinstance(data, list):
            return ", ".join([
                d["name"] for d in data
                if isinstance(d, dict) and "name" in d
            ])
    except:
        pass

    # Case 2: Already comma-separated string
    if isinstance(x, str) and "," in x:
        return ", ".join([i.strip() for i in x.split(",")])

    # Case 3: Single word or "Not Available"
    if isinstance(x, str):
        if x.lower() == "not available":
            return ""
        return x.strip()

    return ""
movie_df["keywords"] = movie_df["keywords"].apply(clean_keywords)
movie_df["keywords"]

In [ ]:
# movie_df["production_companies"] = movie_df["production_companies"].apply(
#     lambda x: ",".join(d["name"] for d in ast.literal_eval(x))
# )
import ast

def clean_production_companies(x):
    # Case 1: JSON-like string
    try:
        data = ast.literal_eval(x)
        if isinstance(data, list):
            return ", ".join([
                d["name"] for d in data
                if isinstance(d, dict) and "name" in d
            ])
    except:
        pass

    # Case 2: Already comma-separated string
    if isinstance(x, str) and "," in x:
        return ", ".join([i.strip() for i in x.split(",")])

    # Case 3: Single value or "Not Available"
    if isinstance(x, str):
        if x.lower() == "not available":
            return ""
        return x.strip()

    return ""
movie_df["production_companies"] = movie_df["production_companies"].apply(clean_production_companies)
movie_df["production_companies"]

In [ ]:
# movie_df["spoken_languages"] = movie_df["spoken_languages"].apply(
#     lambda x: ",".join(d["name"] for d in ast.literal_eval(x))
# )
import ast

def clean_languages(x):
    # Case 1: JSON-like string
    try:
        data = ast.literal_eval(x)
        if isinstance(data, list):
            return ", ".join([
                d["name"] for d in data
                if isinstance(d, dict) and "name" in d
            ])
    except:
        pass

    # Case 2: Already comma-separated string
    if isinstance(x, str) and "," in x:
        return ", ".join([i.strip() for i in x.split(",")])

    # Case 3: Single value or "Not Available"
    if isinstance(x, str):
        if x.lower() == "not available":
            return ""
        return x.strip()

    return ""
movie_df["spoken_languages"] = movie_df["spoken_languages"].apply(clean_languages)
movie_df["spoken_languages"]

In [ ]:
# movie_df["production_countries"] = movie_df["production_countries"].apply(
#     lambda x: ",".join(d["name"] for d in ast.literal_eval(x))
# )
import ast

def clean_countries(x):
    # Case 1: JSON-like string
    try:
        data = ast.literal_eval(x)
        if isinstance(data, list):
            return ", ".join([
                d["name"] for d in data
                if isinstance(d, dict) and "name" in d
            ])
    except:
        pass

    # Case 2: Already comma-separated string
    if isinstance(x, str) and "," in x:
        return ", ".join([i.strip() for i in x.split(",")])

    # Case 3: Single value or "Not Available"
    if isinstance(x, str):
        if x.lower() == "not available":
            return ""
        return x.strip()

    return ""
movie_df["production_countries"] = movie_df["production_countries"].apply(clean_countries)
movie_df["production_countries"]

In [ ]:
# Replace NaN values with an empty string and convert all columns to strings
copymovies_df = movie_df.copy()
movie_df['tags'] = (
    copymovies_df['overview'].fillna('') +
    copymovies_df['genres'].astype(str).fillna('') +
    copymovies_df['keywords'].astype(str).fillna('') +
    copymovies_df['cast'].astype(str).fillna('')
)

print(movie_df['tags'].head())

In [ ]:
import nltk
nltk.download('stopwords')
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

# Initialize stemmer
ps = PorterStemmer()

# Define a function to preprocess text
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()  
    # Remove non-alphanumeric characters
    text = re.sub(r'\W', ' ', text)
    # Tokenize and remove stopwords
    text = ' '.join([word for word in text.split()if word not in stopwords.words('english')])          
    # Apply stemming
    text = ' '.join([ps.stem(word) for word in text.split()])
    return text

# Apply preprocessing to the tags column
movie_df['tags'] = movie_df['tags'].apply(preprocess_text)
movie_df['tags']

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000)  # You can adjust the max_features value as needed

# Transform the tags column
tfidf_matrix = tfidf.fit_transform(movie_df['tags'])

# Save feature names for debugging or exploration
feature_names = tfidf.get_feature_names_out()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity
similarity_matrix = cosine_similarity(tfidf_matrix)

# Save similarity matrix for future use
import pickle
with open("../DATA/PROCESSED/similarity_matrix.pkl", "wb") as file:
    pickle.dump(similarity_matrix, file)

In [ ]:
def recommend_movies(movie_title, movie_df, similarity_matrix):
    # Find the index of the movie
    movie_idx = movies_df[movie_df['original_title'].str.lower() == movie_title.lower()].index[0]
    
    # Get similarity scores for the movie
    similarity_scores = list(enumerate(similarity_matrix[movie_idx]))
   
    # Sort movies by similarity scores (descending) and exclude the first movie (itself)
    similar_movies = sorted(similarity_scores, key=lambda x: x[1], reverse=True)[0:6]
    # Fetch details of recommended movies
    recommendations = []
    for idx, _ in similar_movies:
        movie_details = {
            'title': movie_df.iloc[idx]['original_title'],
            'release_date': movie_df.iloc[idx]['release_date'],
            'genres': movie_df.iloc[idx]['genres'],
            'spoken_languages': movie_df.iloc[idx]['spoken_languages'],
            'tagline': movie_df.iloc[idx]['tagline'],
            'cast': movie_df.iloc[idx]['cast'],
            'crew': movie_df.iloc[idx]['crew'],
        }
        recommendations.append(movie_details)
    
    return recommendations

In [ ]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
import torch

name_search = input("Enter the movie name: ")

model_name = "deepset/roberta-base-squad2"

# Force PyTorch usage
nlp = pipeline('question-answering',
               model=model_name, tokenizer=model_name,framework='pt')

QA_input = {
    'question': 'What is the name of the movie he is talking about?',
    'context': name_search
}
res = nlp(QA_input)["answer"].replace(" movie", "").strip()

print(res)

In [ ]:
from rapidfuzz import process
def find_closest_movie(movie_title):
    """Find the closest matching movie title using rapidfuzz."""
    best_match = process.extractOne(movie_title.lower(), movie_df['original_title'].str.lower(), score_cutoff=80)
    return movie_df.iloc[best_match[2]]['original_title'] if best_match else None

# # for rec in recommend_movies(find_closest_movie(res), movie_df, similarity_matrix):
# print(res)

print(find_closest_movie("Avatur"))

In [ ]:
from rapidfuzz import process, fuzz

def find_closest_movie(movie_title):
    """Find the closest matching movie title using rapidfuzz with improved matching."""
    # Convert input to lowercase for better matching
    movie_title = movie_title.lower().strip()
    
    # Use a lower score cutoff (60) and token_sort_ratio for better partial matches
    best_match = process.extractOne(
        movie_title, 
        movie_df['original_title'].str.lower(),
        scorer=fuzz.token_sort_ratio,
        score_cutoff=60
    )
    
    return movie_df.iloc[best_match[2]]['original_title'] if best_match else None

print(find_closest_movie("Avenger"))

In [ ]:
# ----------------------------
# Imports
# ----------------------------
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
import torch
from rapidfuzz import process, fuzz

# ----------------------------
# Functions
# ----------------------------

# 1️⃣ QA Model function (from your Code 1)
def extract_movie_name_QA(user_input):
    model_name = "deepset/roberta-base-squad2"
    nlp = pipeline('question-answering', model=model_name, tokenizer=model_name, framework='pt')
    
    QA_input = {
        'question': 'What is the name of the movie he is talking about?',
        'context': user_input
    }
    
    res = nlp(QA_input)["answer"].replace(" movie", "").strip()
    return res

# 2️⃣ Improved RapidFuzz matching (from your Code 3)
def find_closest_movie(movie_title, movie_df):
    movie_title = movie_title.lower().strip()
    choices = movie_df['original_title'].str.lower().tolist()
    
    best_match = process.extractOne(
        movie_title,
        choices,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=60
    )
    
    if best_match:
        return movie_df.iloc[best_match[2]]['original_title']
    return None

# 3️⃣ Recommend movies function (from your Code 4)
def recommend_movies(movie_title, movie_df, similarity_matrix, top_n=5):
    try:
        movie_idx = movie_df[movie_df['original_title'].str.lower() == movie_title.lower()].index[0]
    except IndexError:
        return []  # movie not found
    
    similarity_scores = list(enumerate(similarity_matrix[movie_idx]))
    similar_movies = sorted(similarity_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    recommendations = []
    for idx, _ in similar_movies:
        movie_details = {
            'title': movie_df.iloc[idx]['original_title'],
            'release_date': movie_df.iloc[idx]['release_date'],
            'genres': movie_df.iloc[idx]['genres'],
            'spoken_languages': movie_df.iloc[idx]['spoken_languages'],
            'tagline': movie_df.iloc[idx]['tagline'],
            'cast': movie_df.iloc[idx]['cast'],
            'crew': movie_df.iloc[idx]['crew'],
        }
        recommendations.append(movie_details)
    
    return recommendations

# ----------------------------
# Main Pipeline
# ----------------------------

# 1️⃣ User input
user_input = input("Enter your request (e.g., 'Give some movies like avutur'): ")

# 2️⃣ Extract movie name using QA
movie_name_QA = extract_movie_name_QA(user_input)

# 3️⃣ Find closest matching movie in dataframe
movie_name_corrected = find_closest_movie(movie_name_QA, movie_df)

# 4️⃣ Recommend movies
if movie_name_corrected:
    recs = recommend_movies(movie_name_corrected, movie_df, similarity_matrix, top_n=5)
    print(f"\nTop 5 movies like '{movie_name_corrected}':\n")
    for i, m in enumerate(recs, 1):
        print(f"{i}. {m['title']} ({m['release_date']}) - Genres: {m['genres']}")
else:
    print("Sorry, could not find a matching movie!")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Check for GPU availability
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# Load model and tokenizer
model_name = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

# Mood categories
moods = ["joyful", "uplifting", "relaxed", "quirky", "cheerful", "wholesome",
    "romantic", "nostalgic", "thoughtful", "sad", "sentimental", "melancholic",
    "exciting", "adventurous", "intense", "dark", "epic", "suspenseful", "gritty",
    "imaginative", "scary", "chill", "angry", "thought-provoking", "mysterious",
    "fast-paced", "rebellious", "powerful", "heroic",
    "hilarious", "sarcastic", "witty",
    "psychological", "disturbing", "mind-bending", "existential"]

# User input
user_sentence = input("Enter your sentence: ")

# Compute scores for each mood
scores = []
for mood in moods:
    input_data = tokenizer(user_sentence, mood, truncation=True, return_tensors="pt").to(device)
    output = model(input_data["input_ids"])
    prediction = torch.softmax(output["logits"][0], -1).tolist()
    scores.append((mood, prediction[0]))  # Selecting "entailment" probability

# Get the highest scoring mood
predicted_mood = max(scores, key=lambda x: x[1])[0]
print(f"The detected mood is: {predicted_mood}")

In [ ]:
from transformers import pipeline
from difflib import get_close_matches

# -------------------------------
# 1️⃣ movie_df is already defined
# Must have columns: 'original_title', 'genres' (list or comma-separated string), 'release_date' (optional)

# -------------------------------
# 2️⃣ Load emotion classifier
emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    return_all_scores=True
)

# -------------------------------
# 3️⃣ Mood to Genre Mapping
mood_to_genres = {
    "joyful": ["Comedy", "Animation", "Adventure", "Family", "Music", "Romance"],
    "exciting": ["Action", "Thriller", "Adventure", "Crime", "Science Fiction", "Western"],
    "relaxed": ["Drama", "Family", "Comedy", "Music", "TV Movie"],
    "romantic": ["Romance", "Drama", "Comedy", "Music"],
    "sad": ["Drama", "Romance", "Documentary", "War", "History"],
    "scary": ["Horror", "Thriller", "Mystery"],
    "hilarious": ["Comedy", "Family", "Animation"],
    "thoughtful": ["Documentary", "Drama", "History", "Foreign"],
    # Add other moods as needed
}

# -------------------------------
# 4️⃣ Synonyms mapping for single words
mood_synonyms = {
    "happy": "joyful",
    "cheerful": "joyful",
    "excited": "exciting",
    "content": "relaxed",
    "love": "romantic",
    "depressed": "sad",
    "funny": "hilarious",
    "frightened": "scary",
    "spooky": "scary"
}

# -------------------------------
# 5️⃣ Helper: closest mood
def find_closest_mood(mood, available_moods):
    matches = get_close_matches(mood, available_moods, n=1, cutoff=0.5)
    return matches[0] if matches else None

# -------------------------------
# 6️⃣ Detect mood from text
def detect_mood(sentence):
    sentence = sentence.lower().strip()
    
    # Check synonym dictionary first
    if sentence in mood_synonyms:
        return mood_synonyms[sentence]
    
    if sentence in mood_to_genres:
        return sentence

    # Use emotion classifier
    results = emotion_classifier(sentence)[0]
    best_result = max(results, key=lambda x: x["score"])
    detected_mood = best_result["label"].lower()
    
    # Map detected mood to closest custom mood
    if detected_mood not in mood_to_genres:
        detected_mood = find_closest_mood(detected_mood, list(mood_to_genres.keys()))
    
    return detected_mood

# -------------------------------
# 7️⃣ Ensure genres column is a list
def fix_genres_column(df):
    def to_list(g):
        if isinstance(g, list):
            return g
        elif isinstance(g, str):
            # Split comma-separated strings into list
            return [genre.strip() for genre in g.split(',')]
        else:
            return []
    df['genres'] = df['genres'].apply(to_list)

# Fix genres in movie_df
fix_genres_column(movie_df)

# -------------------------------
# 8️⃣ Recommend movies based on mood
def recommend_movies_by_mood(mood, movie_df, top_n=5):
    if not mood:
        print("❌ Could not detect a valid mood.")
        return
    
    genres = mood_to_genres.get(mood)
    if not genres:
        print(f"No exact genres for mood '{mood}', using closest available genres.")
        closest_mood = find_closest_mood(mood, list(mood_to_genres.keys()))
        genres = mood_to_genres.get(closest_mood, [])
    
    recommendations = movie_df[movie_df['genres'].apply(
        lambda g: any(genre in g for genre in genres)
    )].head(top_n)
    
    print(f"\n🎭 Detected Mood: {mood}")
    print(f"🎬 Recommended Genres: {genres}")
    print("\n🍿 Recommended Movies:\n")
    
    if recommendations.empty:
        print("No movies found matching this mood/genres.")
        return
    
    for _, movie in recommendations.iterrows():
        title = movie['original_title']
        genres_str = ', '.join(movie['genres'])
        release = movie.get('release_date', '')
        if release:
            print(f"- {title} ({genres_str}) [{release}]")
        else:
            print(f"- {title} ({genres_str})")

# -------------------------------
# 9️⃣ Main program: get user input
if __name__ == "__main__":
    user_input = input("Tell me how you feel right now: ")
    mood = detect_mood(user_input)
    print(f"\n✅ Detected Mood: {mood}")  # explicitly show mood
    recommend_movies_by_mood(mood, movie_df)


In [ ]:
# Initialize search history as a set
search_history = set()

# Function to recommend movies by cast name using your movie_df
def recommend_by_cast_name(cast_name, movie_df, search_history):
    # Ensure 'original_title' and 'cast' columns exist
    required_columns = {'original_title', 'cast'}
    if not required_columns.issubset(movie_df.columns):
        print(f"movie_df must contain the following columns: {required_columns}")
        return []

    # Filter movies containing the cast name (case-insensitive)
    cast_matches = movie_df[movie_df['cast'].str.contains(cast_name, case=False, na=False)]

    if cast_matches.empty:
        print(f"No movies found with cast name: {cast_name}")
        return []

    # Remove movies already recommended
    cast_matches = cast_matches[~cast_matches['original_title'].isin(search_history)]

    # Update search history
    search_history.update(cast_matches['original_title'].tolist())

    # Return top 5 movies
    return cast_matches['original_title'].head(5).tolist()


# -------------------------------
# Take user input for cast name
if __name__ == "__main__":
    user_cast = input("Enter the name of an actor/actress: ").strip()
    recommended_movies = recommend_by_cast_name(user_cast, movie_df, search_history)

    if recommended_movies:
        print(f"\n🎬 Movies with '{user_cast}':\n")
        for i, title in enumerate(recommended_movies, start=1):
            print(f"{i}. {title}")
    else:
        print("No recommendations found.")


In [ ]:
# ----------------------------
# Imports
# ----------------------------
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ----------------------------
# Load your dataset
# ----------------------------
df = movie_df.copy()

# Clean data
df["original_title"] = df["original_title"].astype(str)
df["tags"] = df["tags"].astype(str).fillna("")

# Remove bad/empty titles
df = df[df["original_title"].str.len() > 1]

# ----------------------------
# Load embedding model (ONLY ONCE)
# ----------------------------
print("🔄 Loading model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# ----------------------------
# Create embeddings (ONLY ONCE)
# ----------------------------
print("⚡ Creating embeddings...")
movie_embeddings = model.encode(
    df["tags"].tolist(),
    show_progress_bar=True
)

print("✅ Model ready!")

# ----------------------------
# Recommendation function
# ----------------------------
def recommend_movies(query, top_n=5):
    query_embedding = model.encode([query])

    similarity = cosine_similarity(query_embedding, movie_embeddings).flatten()

    indices = similarity.argsort()[::-1][:top_n]

    movie_list = df.iloc[indices]["original_title"].tolist()

    return movie_list


# ----------------------------
# Single Run (NO LOOP)
# ----------------------------
query = input("\n🎬 Enter what kind of movie you want: ")

results = recommend_movies(query, top_n=8)

print("\n✨ Recommended Movies:\n")

for i, movie in enumerate(results, 1):
    print(f"{i}. {movie}")

In [ ]:
import pickle

pickle.dump(tfidf, open("../DATA/PROCESSED/tfidf.pkl", "wb"))
pickle.dump(movie_embeddings, open("../DATA/PROCESSED/embeddings.pkl", "wb"))
pickle.dump(movie_df, open("../DATA/PROCESSED/movies.pkl", "wb"))